In [19]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [20]:
max_features = 10000
max_len = 500

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)
print(f'Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}')
print(f'Testing data shape: {X_test.shape}, Testing labels shape: {y_test.shape}')

Training data shape: (25000,), Training labels shape: (25000,)
Testing data shape: (25000,), Testing labels shape: (25000,)


In [21]:
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

decoded_review = ' '.join([reverse_word_index.get(i - 3, '?') for i in X_train[0]])
print(decoded_review)

? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you thi

In [22]:
X_train = sequence.pad_sequences(X_train, maxlen=max_len)
X_test = sequence.pad_sequences(X_test, maxlen=max_len)
print(X_train.shape, X_test.shape)

(25000, 500) (25000, 500)


In [23]:
model = Sequential([
    Embedding(max_features, 128),
    SimpleRNN(128, activation='tanh', dropout=0.2),
    Dense(1, activation='sigmoid')
])

model.build(input_shape=(None, max_len))
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, None, 128)         1280000   
                                                                 
 simple_rnn_2 (SimpleRNN)    (None, 128)               32896     
                                                                 
 dense_2 (Dense)             (None, 1)                 129       
                                                                 
Total params: 1313025 (5.01 MB)
Trainable params: 1313025 (5.01 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [24]:
model.compile(optimizer=Adam(clipnorm=1.0),
              loss='binary_crossentropy',
              metrics=['accuracy'])

earlystopping = EarlyStopping(monitor='val_loss', patience=3,
                              restore_best_weights=True)

In [25]:
history = model.fit(X_train, y_train,
                    epochs=10, batch_size=32,
                    validation_split=0.2,
                    callbacks=[earlystopping])

Epoch 1/10
625/625 [==============================] - 152s 238ms/step - loss: 0.6786 - accuracy: 0.5573 - val_loss: 0.5579 - val_accuracy: 0.7026
Epoch 2/10
625/625 [==============================] - 148s 237ms/step - loss: 0.5146 - accuracy: 0.7533 - val_loss: 0.5738 - val_accuracy: 0.7156
Epoch 3/10
625/625 [==============================] - 145s 231ms/step - loss: 0.4347 - accuracy: 0.8120 - val_loss: 0.4823 - val_accuracy: 0.7990
Epoch 4/10
625/625 [==============================] - 143s 229ms/step - loss: 0.3601 - accuracy: 0.8525 - val_loss: 0.4556 - val_accuracy: 0.7904
Epoch 5/10
625/625 [==============================] - 144s 230ms/step - loss: 0.3200 - accuracy: 0.8727 - val_loss: 0.4940 - val_accuracy: 0.7944
Epoch 6/10
625/625 [==============================] - 144s 230ms/step - loss: 0.2825 - accuracy: 0.8919 - val_loss: 0.4813 - val_accuracy: 0.8254
Epoch 7/10
625/625 [==============================] - 145s 233ms/step - loss: 0.2501 - accuracy: 0.9067 - val_loss: 0.4727 -

In [26]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}')

782/782 [==============================] - 43s 55ms/step - loss: 0.4597 - accuracy: 0.7831
Test loss: 0.4597, Test accuracy: 0.7831


In [27]:
model.save('simple_rnn_imdb.h5')